# Cheap test: do GATED localized features beat the full-buffer baseline?

The oracle proved localization holds +0.10 (one var-ratio stat: full-buffer 0.543 -> oracle
0.641). Naive causal localization FAILED (0.536) because a hard GLR-argmax split fires on
noise — it has no null. The fix under test: give CatBoost the localized statistics **and**
the GLR score as a **gate**, and let the trees learn to trust the localized stat only when
the changepoint evidence is real (a tree splits on the gate, then on the localized stat).

Each online step, causally:
1. scan candidate splits over the buffer [0,t], take the GLR-best split `k*` (variance-change
   likelihood-ratio) and its score,
2. compute statistics on the post-split segment `[k*, t]` vs the reference,
3. emit them + the GLR score (the gate).

Features (5): `loc_logvar`, `loc_mean`, `loc_absmean`, `loc_lenfrac`, `glr_gate`.
Test on the honest 5-fold CV: **CatBoost(50 base) vs CatBoost(50 + these 5)**, `paired_compare`.
A win only at t >= 3 (per [[honest-cv-harness]]: 5-fold t~2 is not safe).

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostClassifier

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
Xbase, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled, base_cols = cache['is_sampled_step'], list(cache['cols'])
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
print(f'base cache: {Xbase.shape} | {len(base_cols)} feats')

base cache: (5036517, 50) | 50 feats


In [2]:
LOC_CACHE = 'loc_features.npz'
LOC_NAMES = ['loc_logvar', 'loc_mean', 'loc_absmean', 'loc_lenfrac', 'glr_gate']
FRACS = np.linspace(0.05, 0.95, 19)
GLR_MIN = 20

def build_series_loc(xo, mu, sd):
    """Causal localized features for one online segment. Row t uses only xo[:t+1]."""
    L = len(xo)
    z = (xo - mu) / sd
    c1 = np.concatenate([[0.0], np.cumsum(z)])
    c2 = np.concatenate([[0.0], np.cumsum(z * z)])
    def var(a, b):
        n = b - a
        if n < 2:
            return 0.0
        m1 = (c1[b] - c1[a]) / n
        return max((c2[b] - c2[a]) / n - m1 * m1, 0.0)
    out = np.zeros((L, 5), np.float32)
    for t in range(L):
        b = t + 1
        kstar, best = 0, 0.0
        if b >= GLR_MIN:
            v0 = var(0, b); lv0 = np.log(v0 + 1e-8)
            ks = (FRACS * b).astype(int)
            for k in ks:
                if k < 8 or k > b - 8:
                    continue
                v1 = var(0, k); v2 = var(k, b)
                if v1 <= 1e-9 or v2 <= 1e-9:
                    continue
                g = b * lv0 - k * np.log(v1 + 1e-8) - (b - k) * np.log(v2 + 1e-8)
                if g > best:
                    best, kstar = g, k
        a = kstar if (b - kstar) >= 2 else 0     # fall back to full buffer if segment too short
        n = b - a; m1 = (c1[b] - c1[a]) / n
        v = max((c2[b] - c2[a]) / n - m1 * m1, 0.0)
        out[t, 0] = np.log(v + 1e-8)             # loc_logvar (z-space: ref var ~ 1)
        out[t, 1] = m1                           # loc_mean
        out[t, 2] = abs(m1)                      # loc_absmean
        out[t, 3] = (b - a) / b                  # loc_lenfrac (share of buffer kept)
        out[t, 4] = best / b                     # glr_gate
    return out

if os.path.exists(LOC_CACHE):
    lc = np.load(LOC_CACHE)
    Xloc, lsid, ltim = lc['X'], lc['sid'], lc['time']
    print('loaded cached localized features', Xloc.shape)
else:
    Xt = pd.read_parquet(os.path.join(TOOLS, '..', '..', '..', 'X_train.parquet'))
    ids = Xt.index.get_level_values('id').to_numpy()
    times = Xt.index.get_level_values('time').to_numpy()
    per = Xt['period'].to_numpy(); val = Xt['value'].to_numpy(np.float64)
    uids, starts = np.unique(ids, return_index=True); bounds = np.append(starts, len(ids))
    Xloc = np.empty((len(sid), 5), np.float32)
    lsid = np.empty(len(sid), np.int64); ltim = np.empty(len(sid), np.int64)
    w = 0; t0 = time.time()
    for k in range(len(uids)):
        s, e = bounds[k], bounds[k + 1]; m = per[s:e] == 2
        ref = val[s:e][~m]; xo = val[s:e][m]; to = times[s:e][m]
        if len(xo) < 1 or (~m).sum() < 8:
            continue                              # same filter build_cache.py used
        mu = float(ref.mean()); sd = max(float(ref.std(ddof=1)), 1e-8)
        L = len(xo)
        Xloc[w:w + L] = build_series_loc(xo, mu, sd)
        lsid[w:w + L] = uids[k]; ltim[w:w + L] = to; w += L
        if (k + 1) % 2000 == 0:
            print(f'  {k + 1:,}/{len(uids):,} | {w:,} rows | {time.time() - t0:.0f}s', flush=True)
    assert w == len(sid), f'row mismatch {w} vs {len(sid)}'
    np.savez(LOC_CACHE, X=Xloc, sid=lsid, time=ltim, cols=np.array(LOC_NAMES))
    print(f'built + saved {LOC_CACHE} in {time.time() - t0:.0f}s')

  2,000/10,000 | 1,001,800 rows | 68s


  4,000/10,000 | 2,021,257 rows | 142s


  6,000/10,000 | 3,033,676 rows | 214s


  8,000/10,000 | 4,034,637 rows | 274s


  10,000/10,000 | 5,036,517 rows | 333s


built + saved loc_features.npz in 333s


In [3]:
# HARD alignment check: localized rows must line up with the base cache row-for-row,
# else the hstack silently mixes series.
assert np.array_equal(lsid, sid), 'sid misalignment vs base cache'
assert np.array_equal(ltim, tim), 'time misalignment vs base cache'
assert np.isfinite(Xloc).all(), 'non-finite localized features'
Xaug = np.hstack([Xbase, Xloc]).astype(np.float32)
aug_cols = base_cols + LOC_NAMES
print(f'aligned OK. augmented matrix {Xaug.shape} ({len(aug_cols)} feats)')
print('localized feature ranges:')
for j, nm in enumerate(LOC_NAMES):
    print(f'  {nm:12s} [{Xloc[:, j].min():.3f}, {Xloc[:, j].max():.3f}] mean {Xloc[:, j].mean():.3f}')

aligned OK. augmented matrix (5036517, 55) (55 feats)
localized feature ranges:


  loc_logvar   [-18.421, 17.904] mean -0.205
  loc_mean     [-10.288, 6319.728] mean 0.032
  loc_absmean  [0.000, 6319.728] mean 0.194
  loc_lenfrac  [0.050, 1.000] mean 0.518
  glr_gate     [0.000, 14.850] mean 0.070


In [4]:
CATP = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
            random_seed=0, verbose=0, thread_count=-1)
folds = CV.series_folds(sid, n_splits=5, seed=0)

def fp(matrix):
    def _f(train, val):
        m = CatBoostClassifier(**CATP)
        m.fit(train.X, train.y, sample_weight=train.w)
        return m.predict_proba(val.X)[:, 1]
    return _f

res = {}
for name, M in [('base_50', Xbase), ('base_plus_loc', Xaug)]:
    t = time.time(); print(f'\n=== {name} ({M.shape[1]} feats) ===', flush=True)
    res[name] = CV.run_cv(M, y, sid, index, fp(M), folds=folds,
                          train_row_mask=sampled, row_step=step)
    json.dump({k: dict(fold_scores=v.fold_scores.tolist(), mean=v.mean, oof=v.oof_score)
               for k, v in res.items()}, open('gated_results.json', 'w'), indent=2)
    print(f'{res[name].summary(name)} | {time.time() - t:.0f}s', flush=True)


=== base_50 (50 feats) ===


  fold 0: train   263,351 rows / 8,000 series | val 1,012,996 rows / 2,000 series | TS-AUC 0.5849


  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5964


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5828


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6014


...

  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5997


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5846


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6052


  fold 4: train   263,109 rows / 8,000 series | val 1,007,487 rows / 2,000 series | TS-AUC 0.6221


base_plus_loc      mean 0.5993 +/- 0.0157 (sem 0.0070) | OOF 0.5983 | folds: 0.5848  0.5997  0.5846  0.6052  0.6221 | 332s


In [5]:
print(CV.summarize(res))
print('\n=== does adding gated localized features help? ===')
print(CV.paired_compare(res['base_plus_loc'], res['base_50'], 'base_plus_loc', 'base_50'))

# where does the model rank the localized features? (are they even used?)
m = CatBoostClassifier(**CATP)
tr = sampled
m.fit(Xaug[tr], y[tr], sample_weight=CV.equal_series_weights(sid[tr]))
imp = pd.Series(m.get_feature_importance(), index=aug_cols).sort_values(ascending=False)
print('\ntop 15 features by importance (localized marked *):')
for nm, v in imp.head(15).items():
    print(f'  {"*" if nm in LOC_NAMES else " "} {nm:20s} {v:.2f}')
print('\nlocalized feature ranks:',
      {nm: int(imp.index.get_loc(nm)) + 1 for nm in LOC_NAMES})

base_plus_loc      mean 0.5993 +/- 0.0157 (sem 0.0070) | OOF 0.5983 | folds: 0.5848  0.5997  0.5846  0.6052  0.6221
base_50            mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232

=== does adding gated localized features help? ===
base_plus_loc - base_50: +0.0016 +/- 0.0021 (sem 0.0009, t +1.64, wins 3/5) -> within noise
  per-fold deltas: -0.0001  +0.0033  +0.0018  +0.0039  -0.0011



top 15 features by importance (localized marked *):
    n_online             12.91
    ref_kurt             9.81
    ref_skew             8.85
    ref_log_n            8.41
    ref_ac1              6.30
    spec_entropy_shift   5.34
    spec_centroid_shift  3.84
    ac2_shift            2.37
    ar10_resid_logratio  2.30
    ac5_shift            2.26
    kuiper               2.15
    kurt_shift           1.63
    cvm                  1.53
    spec_entropy         1.53
    ar2_resid_logratio   1.53

localized feature ranks: {'loc_logvar': 36, 'loc_mean': 49, 'loc_absmean': 51, 'loc_lenfrac': 48, 'glr_gate': 22}
